# release16_joint_drawdown_release_target_contract_v1

Purpose: build a fold-local joint target contract that addresses the release16 sparse-slice root cause before any new retained training run.

Baselines:
- Canonical baseline: `v19reg_v22_budget_usage_bull_guard_v1`.
- Direct comparison baseline: `release16_nonconvex_checkpoint_bc_official_replay_v1`.
- Source diagnostic: `release16_sparse_slice_fit_root_cause_v1`.
- Source target contract: `release16_failedfold_state_head_target_recipe_v1/state_head_target_contract.csv`.

Role: diagnose and prepare a target-revision handoff.

Changed layer: target, model_selection, bc_warm_start.

Target window: `wf_2020`, `wf_2022`, `wf_2023_2024`; no actor can be retained from this card.

Hypothesis: strict fold-local drawdown-control rows repair the missing drawdown target coverage, while an auxiliary state-head release candidate is required to repair sparse release/rebound label coverage in the guard fold. Promotion remains blocked until a downstream BC holdout/replay confirms slice fit and walk-forward path metrics.

Acceptance criteria:
- Fold-local scope audit has zero eval overlap.
- Drawdown-control rows are present in `wf_2022_train` and `wf_2023_2024_train`.
- Release/rebound/high-participation rows reach the sparse-slice minimum in affected folds for the selected contract variant.
- Lift audit passes under the same `state_objective_target_position_from_row()` risk-priority semantics consumed by BC labels.
- Label distribution passes the existing BC readiness gate.

Closest success: `release16_failedfold_state_head_target_recipe_v1` produced a fold-local state-head recipe with BC-ready label distribution and zero guard targets.

Closest failure: `release16_sparse_slice_fit_root_cause_v1` showed missing drawdown targets, sparse release rows, action-head collapse, and path risk in `wf_2020`.

Why this is not a duplicate: this notebook composes fold-local drawdown-control targets with strict and auxiliary release/state-head variants, then audits coverage and trainability before any replay or training.


In [1]:

from __future__ import annotations

from datetime import datetime, timezone
from pathlib import Path
import json
import math
import os
import sys
from typing import Any

import numpy as np
import pandas as pd

ROOT = next(path for path in [Path.cwd(), *Path.cwd().parents] if (path / 'src' / 'soxl_quant').exists())
os.chdir(ROOT)
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from src.soxl_quant.app.state_objective_training_targets import (
    StateObjectiveBCReadinessConfig,
    StateObjectiveDrawdownControlTargetConfig,
    audit_state_objective_target_lift,
    audit_state_objective_target_scope,
    build_drawdown_control_training_targets,
    build_fold_local_joint_target_contract,
    score_state_objective_bc_readiness,
    summarize_state_objective_bc_readiness,
    summarize_state_objective_target_lift_audit,
    summarize_state_objective_target_scope_audit,
)
from src.soxl_quant.policy.policy_builder import state_objective_target_position_from_row

EXPERIMENT = 'release16_joint_drawdown_release_target_contract_v1'
CARD = ROOT / 'runtime' / 'model_selection' / 'card' / EXPERIMENT
CARD.mkdir(parents=True, exist_ok=True)
SOURCE_CARD = 'release16_failedfold_state_head_target_recipe_v1'
SPARSE_SOURCE_CARD = 'release16_sparse_slice_fit_root_cause_v1'
SOURCE_CONTRACT = ROOT / 'runtime' / 'model_selection' / 'card' / SOURCE_CARD / 'state_head_target_contract.csv'
SPARSE_SUMMARY_PATH = ROOT / 'runtime' / 'model_selection' / 'card' / SPARSE_SOURCE_CARD / 'summary.json'
REPAIR_REQUIREMENTS_PATH = ROOT / 'runtime' / 'model_selection' / 'card' / SPARSE_SOURCE_CARD / 'repair_requirements.csv'
AFFECTED_FOLDS = ['wf_2022_train', 'wf_2023_2024_train']
MIN_SPARSE_ROWS = 20
TARGET_MAX_DD_PCT = 20.0
DRAWNDOWN_STATE_THRESHOLD = 0.10
DRAWDOWN_CAP = 0.25


def read_json(path: Path) -> dict[str, Any]:
    return json.loads(path.read_text(encoding='utf-8')) if path.exists() else {}


def json_safe(value: Any) -> Any:
    if isinstance(value, dict):
        return {str(k): json_safe(v) for k, v in value.items()}
    if isinstance(value, (list, tuple, set)):
        return [json_safe(v) for v in value]
    if isinstance(value, (np.integer,)):
        return int(value)
    if isinstance(value, (np.floating,)):
        f = float(value)
        return f if math.isfinite(f) else None
    if isinstance(value, (np.bool_,)):
        return bool(value)
    if isinstance(value, pd.Timestamp):
        return value.isoformat()
    if isinstance(value, float):
        return value if math.isfinite(value) else None
    return value


def write_json(path: Path, payload: dict[str, Any]) -> None:
    path.write_text(json.dumps(json_safe(payload), indent=2, sort_keys=True) + '\n', encoding='utf-8')


def bool_series(frame: pd.DataFrame, column: str) -> pd.Series:
    if column not in frame.columns:
        return pd.Series(False, index=frame.index)
    series = frame[column]
    if series.dtype == bool:
        return series.fillna(False)
    return series.fillna(False).map(lambda value: str(value).strip().lower() in {'1', 'true', 'yes', 'y'})

source = pd.read_csv(SOURCE_CONTRACT)
source['datetime'] = pd.to_datetime(source['datetime']).dt.normalize()
for col in ['train_start', 'train_end', 'eval_start', 'eval_end']:
    source[col] = pd.to_datetime(source[col]).dt.normalize().dt.strftime('%Y-%m-%d')
for col in ['is_rebound_cluster', 'is_downtrend_state', 'release_gate_state', 'drawdown_control_state', 'failedfold_state_head_target_state']:
    if col in source.columns:
        source[col] = bool_series(source, col)

sparse_summary = read_json(SPARSE_SUMMARY_PATH)
repair_requirements = pd.read_csv(REPAIR_REQUIREMENTS_PATH) if REPAIR_REQUIREMENTS_PATH.exists() else pd.DataFrame()
print({'source_rows': len(source), 'source_folds': source['fold'].dropna().unique().tolist(), 'sparse_decision': sparse_summary.get('decision')})


{'source_rows': 3731, 'source_folds': ['wf_2020_train', 'wf_2022_train', 'wf_2023_2024_train'], 'sparse_decision': 'release16_sparse_slice_target_repair_required_no_training_no_promotion'}


Gym has been unmaintained since 2022 and does not support NumPy 2.0 amongst other critical functionality.
Please upgrade to Gymnasium, the maintained drop-in replacement of Gym, or contact the authors of your software and request that they upgrade.
Users of this version of Gym should be able to simply replace 'import gym' with 'import gymnasium as gym' in the vast majority of cases.
See the migration guide at https://gymnasium.farama.org/introduction/migration_guide/ for additional information.


In [2]:

folds: list[dict[str, str]] = []
for fold_name, group in source.groupby('fold', sort=True):
    folds.append(
        {
            'name': str(fold_name),
            'train_start': str(group['train_start'].iloc[0]),
            'train_end': str(group['train_end'].iloc[0]),
            'eval_start': str(group['eval_start'].iloc[0]),
            'eval_end': str(group['eval_end'].iloc[0]),
            'source_window': str(fold_name),
        }
    )

source_bool_cols = ['is_rebound_cluster', 'is_downtrend_state', 'release_gate_state', 'drawdown_control_state', 'failedfold_state_head_target_state']
source_profile = (
    source.assign(**{col: bool_series(source, col) for col in source_bool_cols if col in source.columns})
    .groupby('fold', dropna=False)
    .agg(
        rows=('datetime', 'size'),
        start=('datetime', 'min'),
        end=('datetime', 'max'),
        release_gate_rows=('release_gate_state', 'sum'),
        state_head_target_rows=('failedfold_state_head_target_state', 'sum'),
        rebound_rows=('is_rebound_cluster', 'sum'),
        downtrend_rows=('is_downtrend_state', 'sum'),
        drawdown_control_rows=('drawdown_control_state', 'sum'),
        max_objective_weight=('objective_weight', 'max'),
    )
    .reset_index()
)
source_profile['release_or_state_head_rows'] = source_profile['release_gate_rows'] + source_profile['state_head_target_rows']
source_profile.to_csv(CARD / 'source_target_coverage.csv', index=False)
print(source_profile.to_string(index=False))
print(pd.DataFrame(folds).to_string(index=False))


              fold  rows      start        end  release_gate_rows  state_head_target_rows  rebound_rows  downtrend_rows  drawdown_control_rows  max_objective_weight  release_or_state_head_rows
     wf_2020_train  2470 2010-03-11 2019-12-31                 42                    1058          1093             781                    781             25.000000                        1100
     wf_2022_train   505 2020-01-02 2021-12-31                  9                       0            13             234                      0             25.000000                           9
wf_2023_2024_train   756 2020-01-02 2022-12-30                 11                     257           266             478                      0             19.648219                         268
              name train_start  train_end eval_start   eval_end      source_window
     wf_2020_train  2010-03-11 2019-12-31 2020-01-01 2020-12-31      wf_2020_train
     wf_2022_train  2020-01-02 2021-12-31 2022-01-03 2022-12-3

In [3]:

def drawdown_intervals_from_nav(frame: pd.DataFrame, *, threshold: float) -> pd.DataFrame:
    nav = pd.to_numeric(frame['strategy_nav'], errors='coerce').ffill().fillna(0.0)
    peak = nav.cummax().replace(0.0, np.nan)
    dd = ((peak - nav) / peak).fillna(0.0).clip(lower=0.0)
    dates = pd.to_datetime(frame['datetime']).dt.normalize()
    mask = dd >= float(threshold)
    rows: list[dict[str, Any]] = []
    rank = 1
    start_idx: int | None = None
    for idx, active in enumerate(mask.tolist() + [False]):
        if active and start_idx is None:
            start_idx = idx
        elif not active and start_idx is not None:
            end_idx = idx - 1
            interval_dd = dd.iloc[start_idx : end_idx + 1]
            rows.append(
                {
                    'rank': rank,
                    'start': dates.iloc[start_idx].strftime('%Y-%m-%d'),
                    'end': dates.iloc[end_idx].strftime('%Y-%m-%d'),
                    'max_dd': float(interval_dd.max() * 100.0),
                }
            )
            rank += 1
            start_idx = None
    return pd.DataFrame(rows)


drawdown_config = StateObjectiveDrawdownControlTargetConfig(
    target_max_dd=TARGET_MAX_DD_PCT,
    drawdown_state_threshold=DRAWNDOWN_STATE_THRESHOLD,
    downtrend_position_cap=DRAWDOWN_CAP,
    min_position_delta=0.02,
    drawdown_gap_weight=10.0,
    interval_weight=5.0,
    max_objective_weight=25.0,
)

passthrough_cols = [
    'datetime', 'teacher_position', 'strategy_nav', 'price', 'base_position', 'upside_position',
    'router_position', 'mom_5', 'mom_21', 'drawdown_63', 'rv_21', 'vol_of_vol_21',
    'soxl_soxx_ratio_widen_21', 'soxl_soxx_ratio_z_252', 'soxx_mom_21'
]
drawdown_frames: list[pd.DataFrame] = []
interval_frames: list[pd.DataFrame] = []
for fold in folds:
    fold_name = fold['name']
    fold_source = source[source['fold'].astype(str) == fold_name].sort_values('datetime').reset_index(drop=True)
    daily_cols = [col for col in passthrough_cols if col in fold_source.columns]
    daily = fold_source[daily_cols].copy()
    daily['position'] = pd.to_numeric(daily['teacher_position'], errors='coerce').fillna(0.0)
    intervals = drawdown_intervals_from_nav(daily, threshold=TARGET_MAX_DD_PCT / 100.0)
    if not intervals.empty:
        tmp = intervals.copy()
        tmp.insert(0, 'fold', fold_name)
        interval_frames.append(tmp)
    targets = build_drawdown_control_training_targets(
        daily,
        intervals=intervals,
        candidate='release16_foldlocal_drawdown_control',
        window=fold_name,
        config=drawdown_config,
    )
    if targets.empty:
        continue
    targets['fold'] = fold_name
    targets['target_scope'] = 'fold_local'
    targets['train_start'] = fold['train_start']
    targets['train_end'] = fold['train_end']
    targets['eval_start'] = fold['eval_start']
    targets['eval_end'] = fold['eval_end']
    drawdown_frames.append(targets)

drawdown_targets = pd.concat(drawdown_frames, ignore_index=True) if drawdown_frames else pd.DataFrame()
drawdown_targets.to_csv(CARD / 'drawdown_control_training_targets.csv', index=False)
drawdown_intervals = pd.concat(interval_frames, ignore_index=True) if interval_frames else pd.DataFrame(columns=['fold', 'rank', 'start', 'end', 'max_dd'])
drawdown_intervals.to_csv(CARD / 'drawdown_control_intervals.csv', index=False)
print(drawdown_targets.groupby('fold').agg(rows=('datetime', 'size'), drawdown_control_rows=('drawdown_control_state', 'sum'), downtrend_rows=('is_downtrend_state', 'sum'), max_path_drawdown=('path_drawdown', 'max'), max_objective_weight=('objective_weight', 'max')).to_string())
print(drawdown_intervals.to_string(index=False))


                    rows  drawdown_control_rows  downtrend_rows  max_path_drawdown  max_objective_weight
fold                                                                                                    
wf_2020_train       2470                    934             928           0.488176                  25.0
wf_2022_train        505                     25               4           0.145760                  25.0
wf_2023_2024_train   756                     35               5           0.145760                  25.0
         fold  rank      start        end    max_dd
wf_2020_train     1 2010-06-07 2010-06-09 24.315242
wf_2020_train     2 2010-06-29 2010-07-06 27.142618
wf_2020_train     3 2010-08-10 2010-09-27 42.121663
wf_2020_train     4 2010-10-01 2010-10-04 23.468000
wf_2020_train     5 2010-10-06 2010-10-07 21.665865
wf_2020_train     6 2011-06-08 2011-06-30 27.031926
wf_2020_train     7 2011-07-11 2012-01-31 48.817577
wf_2020_train     8 2012-03-05 2012-03-07 23.681476
wf_2020

In [4]:

def enrich_joint_contract(contract: pd.DataFrame, *, variant: str, aux_dates: pd.DataFrame | None = None) -> pd.DataFrame:
    if contract.empty:
        result = contract.copy()
        result['contract_variant'] = variant
        return result
    result = contract.copy()
    result['datetime'] = pd.to_datetime(result['datetime']).dt.normalize()
    meta_cols = [
        'fold', 'datetime', 'action', 'price', 'strategy_nav', 'buyhold_nav',
        'failedfold_state_head_target_state', 'failedfold_state_head_target_lift',
        'failedfold_state_head_feature_hits', 'failedfold_state_head_valid_features',
        'failedfold_state_head_guard_fold', 'failedfold_state_head_target_position',
    ]
    meta_cols = [col for col in meta_cols if col in source.columns]
    meta = source[meta_cols].copy().drop_duplicates(['fold', 'datetime'], keep='last')
    result = result.merge(meta, on=['fold', 'datetime'], how='left')
    for col in ['release_gate_state', 'drawdown_control_state', 'is_rebound_cluster', 'is_downtrend_state', 'failedfold_state_head_target_state']:
        result[col] = bool_series(result, col)
    result['aux_state_head_release_state'] = False
    if aux_dates is not None and not aux_dates.empty:
        keys = set(zip(aux_dates['fold'].astype(str), pd.to_datetime(aux_dates['datetime']).dt.normalize()))
        result['aux_state_head_release_state'] = [
            (str(row.fold), pd.Timestamp(row.datetime).normalize()) in keys
            for row in result[['fold', 'datetime']].itertuples(index=False)
        ]
    for col in ['failedfold_state_head_target_lift', 'failedfold_state_head_feature_hits', 'failedfold_state_head_valid_features', 'failedfold_state_head_target_position']:
        if col in result.columns:
            result[col] = pd.to_numeric(result[col], errors='coerce').fillna(0.0)
    if 'action' not in result.columns:
        result['action'] = 'HOLD'
    result['action'] = result['action'].fillna('HOLD').astype(str)
    result['effective_target_position'] = [state_objective_target_position_from_row(row) for row in result.to_dict('records')]
    result['contract_variant'] = variant
    return result.sort_values(['fold', 'datetime']).reset_index(drop=True)


strict_joint = build_fold_local_joint_target_contract(
    source,
    drawdown_targets,
    folds=folds,
    project_auxiliary_train_rows=False,
)
strict_joint = enrich_joint_contract(strict_joint, variant='strict_joint')
strict_joint.to_csv(CARD / 'strict_joint_target_contract.csv', index=False)
print(strict_joint.groupby('fold').agg(rows=('datetime', 'size'), release=('release_gate_state', 'sum'), state_head=('failedfold_state_head_target_state', 'sum'), drawdown=('drawdown_control_state', 'sum'), downtrend=('is_downtrend_state', 'sum')).to_string())


                    rows  release  state_head  drawdown  downtrend
fold                                                              
wf_2020_train       2470       42        1058       976        970
wf_2022_train        505        9           0        25        236
wf_2023_2024_train   756       11         257        35        480


In [5]:

feature_hits = pd.to_numeric(source.get('failedfold_state_head_feature_hits', pd.Series(0, index=source.index)), errors='coerce').fillna(0).astype(int)
valid_features = pd.to_numeric(source.get('failedfold_state_head_valid_features', pd.Series(0, index=source.index)), errors='coerce').fillna(0).astype(int)
state_lift = pd.to_numeric(source.get('failedfold_state_head_target_lift', pd.Series(0.0, index=source.index)), errors='coerce').fillna(0.0)
state_target_pos = pd.to_numeric(source.get('failedfold_state_head_target_position', pd.Series(np.nan, index=source.index)), errors='coerce')
aux_mask = (
    source['target_scope'].fillna('').astype(str).eq('fold_local')
    & source['fold'].astype(str).isin(AFFECTED_FOLDS)
    & (valid_features >= 3)
    & (feature_hits >= 3)
    & (state_lift > 0.02)
)
aux_state_head_targets = source.loc[aux_mask].copy()
if not aux_state_head_targets.empty:
    target_position = state_target_pos.loc[aux_state_head_targets.index].fillna(
        pd.to_numeric(aux_state_head_targets['risk_target_position_original'], errors='coerce').fillna(0.0)
        + state_lift.loc[aux_state_head_targets.index]
    ).clip(0.0, 1.0)
    aux_state_head_targets['candidate'] = 'release16_aux_state_head_release_target'
    aux_state_head_targets['window'] = aux_state_head_targets['fold'].astype(str)
    aux_state_head_targets['target_scope'] = 'fold_local'
    aux_state_head_targets['rebound_target_position'] = np.maximum(
        pd.to_numeric(aux_state_head_targets['rebound_target_position'], errors='coerce').fillna(0.0),
        target_position,
    )
    aux_state_head_targets['risk_target_position_original'] = pd.to_numeric(
        aux_state_head_targets['risk_target_position_original'], errors='coerce'
    ).fillna(pd.to_numeric(aux_state_head_targets['risk_target_position'], errors='coerce'))
    aux_state_head_targets['risk_target_position'] = np.maximum(
        pd.to_numeric(aux_state_head_targets['risk_target_position'], errors='coerce').fillna(0.0),
        target_position,
    )
    aux_state_head_targets['is_rebound_cluster'] = True
    aux_state_head_targets['release_gate_state'] = True
    aux_state_head_targets['aux_state_head_release_state'] = True
    aux_state_head_targets['objective_weight'] = np.maximum(
        pd.to_numeric(aux_state_head_targets['objective_weight'], errors='coerce').fillna(1.0),
        (1.0 + state_lift.loc[aux_state_head_targets.index] * 60.0).clip(upper=25.0),
    )
    aux_state_head_targets['target_recipe'] = aux_state_head_targets.get('target_recipe', '').fillna('').astype(str) + ',aux_state_head_release_candidate'

aux_cols = [col for col in drawdown_targets.columns if col in aux_state_head_targets.columns]
aux_targets = pd.concat(
    [drawdown_targets, aux_state_head_targets[aux_cols] if aux_cols else pd.DataFrame()],
    ignore_index=True,
    sort=False,
)
aux_state_head_targets.to_csv(CARD / 'aux_state_head_release_targets.csv', index=False)
aux_targets.to_csv(CARD / 'joint_auxiliary_targets.csv', index=False)

aux_joint = build_fold_local_joint_target_contract(
    source,
    aux_targets,
    folds=folds,
    project_auxiliary_train_rows=False,
)
aux_joint = enrich_joint_contract(aux_joint, variant='aux_state_head_joint', aux_dates=aux_state_head_targets)
aux_joint.to_csv(CARD / 'joint_target_contract.csv', index=False)
print(aux_state_head_targets.groupby('fold').agg(rows=('datetime', 'size'), mean_lift=('failedfold_state_head_target_lift', 'mean'), max_weight=('objective_weight', 'max')).to_string())
print(aux_joint.groupby('fold').agg(rows=('datetime', 'size'), release=('release_gate_state', 'sum'), aux_release=('aux_state_head_release_state', 'sum'), state_head=('failedfold_state_head_target_state', 'sum'), drawdown=('drawdown_control_state', 'sum'), downtrend=('is_downtrend_state', 'sum')).to_string())


                    rows  mean_lift  max_weight
fold                                           
wf_2022_train        338   0.331938        25.0
wf_2023_2024_train   455   0.356786        25.0
                    rows  release  aux_release  state_head  drawdown  downtrend
fold                                                                           
wf_2020_train       2470       42            0        1058       976        970
wf_2022_train        505        9          338           0        25        236
wf_2023_2024_train   756       11          455         257        35        480


In [6]:

def classify_target_action(row: pd.Series) -> str:
    eps = 1e-9
    teacher = float(row.get('teacher_position', 0.0) or 0.0)
    risk = float(row.get('risk_target_position', teacher) or teacher)
    rebound = float(row.get('rebound_target_position', teacher) or teacher)
    if bool(row.get('drawdown_control_state', False)) and bool(row.get('is_downtrend_state', False)) and risk < teacher - eps:
        return 'DRAWDOWN_CAP'
    if bool(row.get('aux_state_head_release_state', False)):
        return 'AUX_STATE_HEAD_LIFT'
    if bool(row.get('release_gate_state', False)) and bool(row.get('is_rebound_cluster', False)) and risk > float(row.get('risk_target_position_original', teacher) or teacher) + eps:
        return 'RELEASE_GATE_LIFT'
    if bool(row.get('failedfold_state_head_target_state', False)):
        return 'STATE_HEAD_LIFT'
    if bool(row.get('is_rebound_cluster', False)) and rebound > teacher + eps:
        return 'REBOUND_LIFT'
    return str(row.get('action') or 'HOLD')


def coverage_table(contract: pd.DataFrame, *, variant: str) -> pd.DataFrame:
    frame = contract.copy()
    for col in ['release_gate_state', 'aux_state_head_release_state', 'failedfold_state_head_target_state', 'drawdown_control_state', 'is_rebound_cluster', 'is_downtrend_state']:
        frame[col] = bool_series(frame, col)
    frame['target_action'] = frame.apply(classify_target_action, axis=1)
    grouped = (
        frame.groupby('fold', dropna=False)
        .agg(
            rows=('datetime', 'size'),
            release_gate_rows=('release_gate_state', 'sum'),
            aux_state_head_release_rows=('aux_state_head_release_state', 'sum'),
            state_head_target_rows=('failedfold_state_head_target_state', 'sum'),
            rebound_rows=('is_rebound_cluster', 'sum'),
            downtrend_rows=('is_downtrend_state', 'sum'),
            drawdown_control_rows=('drawdown_control_state', 'sum'),
            effective_target_min=('effective_target_position', 'min'),
            effective_target_max=('effective_target_position', 'max'),
            effective_target_mean=('effective_target_position', 'mean'),
        )
        .reset_index()
    )
    grouped['contract_variant'] = variant
    grouped['release_or_aux_rows'] = grouped['release_gate_rows'] + grouped['aux_state_head_release_rows'] + grouped['state_head_target_rows']
    grouped['drawdown_coverage_gate'] = grouped.apply(
        lambda row: True if row['fold'] not in AFFECTED_FOLDS else int(row['drawdown_control_rows']) >= MIN_SPARSE_ROWS,
        axis=1,
    )
    grouped['release_coverage_gate'] = grouped.apply(
        lambda row: True if row['fold'] not in AFFECTED_FOLDS else int(row['release_or_aux_rows']) >= MIN_SPARSE_ROWS,
        axis=1,
    )
    grouped['coverage_gate'] = grouped['drawdown_coverage_gate'] & grouped['release_coverage_gate']
    return grouped


def action_distribution(contract: pd.DataFrame, *, variant: str) -> pd.DataFrame:
    frame = contract.copy()
    for col in ['release_gate_state', 'aux_state_head_release_state', 'failedfold_state_head_target_state', 'drawdown_control_state', 'is_rebound_cluster', 'is_downtrend_state']:
        frame[col] = bool_series(frame, col)
    frame['target_action'] = frame.apply(classify_target_action, axis=1)
    rows = []
    for fold, group in frame.groupby('fold', dropna=False, sort=True):
        counts = group['target_action'].value_counts(dropna=False).rename_axis('target_action').reset_index(name='rows')
        counts['share'] = counts['rows'] / len(group)
        counts.insert(0, 'fold', fold)
        counts.insert(0, 'contract_variant', variant)
        rows.append(counts)
    overall = frame['target_action'].value_counts(dropna=False).rename_axis('target_action').reset_index(name='rows')
    overall['share'] = overall['rows'] / len(frame)
    overall.insert(0, 'fold', 'all')
    overall.insert(0, 'contract_variant', variant)
    rows.append(overall)
    return pd.concat(rows, ignore_index=True)


def bc_profile_rows(distribution: pd.DataFrame) -> pd.DataFrame:
    rows = []
    for variant, group in distribution[distribution['fold'] == 'all'].groupby('contract_variant', sort=True):
        shares = pd.to_numeric(group['share'], errors='coerce').fillna(0.0)
        entropy = float(-(shares[shares > 0] * np.log(shares[shares > 0])).sum())
        rows.append(
            {
                'profile': str(variant),
                'samples': int(pd.to_numeric(group['rows'], errors='coerce').fillna(0).sum()),
                'unique_actions': int((pd.to_numeric(group['rows'], errors='coerce').fillna(0) > 0).sum()),
                'dominant_action_share': float(shares.max()) if len(shares) else 1.0,
                'action_entropy': entropy,
            }
        )
    return pd.DataFrame(rows)

coverage = pd.concat([
    coverage_table(strict_joint, variant='strict_joint'),
    coverage_table(aux_joint, variant='aux_state_head_joint'),
], ignore_index=True)
coverage.to_csv(CARD / 'joint_target_coverage.csv', index=False)

distribution = pd.concat([
    action_distribution(strict_joint, variant='strict_joint'),
    action_distribution(aux_joint, variant='aux_state_head_joint'),
], ignore_index=True)
distribution.to_csv(CARD / 'bc_label_distribution.csv', index=False)

bc_input = bc_profile_rows(distribution)
bc_readiness = score_state_objective_bc_readiness(
    bc_input,
    target_scope='fold_local',
    config=StateObjectiveBCReadinessConfig(max_dominant_action_share=0.90, min_unique_actions=3, min_samples=100),
)
bc_readiness.to_csv(CARD / 'bc_readiness_scores.csv', index=False)

print(coverage.to_string(index=False))
print(distribution[distribution['fold'].eq('all')].to_string(index=False))
print(bc_readiness.to_string(index=False))


              fold  rows  release_gate_rows  aux_state_head_release_rows  state_head_target_rows  rebound_rows  downtrend_rows  drawdown_control_rows  effective_target_min  effective_target_max  effective_target_mean     contract_variant  release_or_aux_rows  drawdown_coverage_gate  release_coverage_gate  coverage_gate
     wf_2020_train  2470                 42                            0                    1058          1093             970                    976              0.052000                0.9151               0.374765         strict_joint                 1100                    True                   True           True
     wf_2022_train   505                  9                            0                       0            13             236                     25              0.047588                1.0000               0.197100         strict_joint                    9                    True                  False          False
wf_2023_2024_train   756             

In [7]:

audit_rows = []
summary_rows = []
for variant, contract in [('strict_joint', strict_joint), ('aux_state_head_joint', aux_joint)]:
    scope_audit = audit_state_objective_target_scope(contract, folds=folds)
    scope_audit.insert(0, 'contract_variant', variant)
    scope_summary = summarize_state_objective_target_scope_audit(scope_audit.drop(columns=['contract_variant']))

    lift_audit = audit_state_objective_target_lift(
        contract,
        group_cols=('fold',),
        baseline_risk_position_col='risk_target_position_original',
    )
    lift_audit.insert(0, 'contract_variant', variant)
    lift_summary = summarize_state_objective_target_lift_audit(lift_audit.drop(columns=['contract_variant']))

    audit_rows.append(('scope', scope_audit))
    audit_rows.append(('lift', lift_audit))
    summary_rows.append(
        {
            'contract_variant': variant,
            'scope_decision': scope_summary['decision'],
            'scope_training_allowed': bool(scope_summary['training_allowed']),
            'scope_eval_overlap_rows': int(scope_summary['eval_overlap_rows']),
            'lift_decision': lift_summary['decision'],
            'lift_training_allowed': bool(lift_summary['training_allowed']),
            'lift_current_positive_rows': int(lift_summary['current_positive_rebound_lift_rows']),
            'lift_rebound_rows': int(lift_summary['rebound_rows']),
        }
    )

scope_audits = pd.concat([frame for kind, frame in audit_rows if kind == 'scope'], ignore_index=True)
lift_audits = pd.concat([frame for kind, frame in audit_rows if kind == 'lift'], ignore_index=True)
audit_summary = pd.DataFrame(summary_rows)
scope_audits.to_csv(CARD / 'target_scope_audit.csv', index=False)
lift_audits.to_csv(CARD / 'target_lift_audit.csv', index=False)
audit_summary.to_csv(CARD / 'target_audit_summary.csv', index=False)
print(audit_summary.to_string(index=False))
print(lift_audits.to_string(index=False))


    contract_variant                         scope_decision  scope_training_allowed  scope_eval_overlap_rows                              lift_decision  lift_training_allowed  lift_current_positive_rows  lift_rebound_rows
        strict_joint state_objective_targets_training_ready                    True                        0 state_objective_target_lift_training_ready                   True                        1112               1372
aux_state_head_joint state_objective_targets_training_ready                    True                        0 state_objective_target_lift_training_ready                   True                        1376               1903
    contract_variant               fold  rows  rebound_rows  downtrend_rows  rebound_downtrend_rows  current_positive_rebound_lift_rows  candidate_release_positive_lift_rows  mean_current_effective_lift  mean_candidate_release_lift  max_current_effective_lift  max_candidate_release_lift  lift_semantics_gate  release_candidate_gate t

In [8]:

source_cov = source_profile[['fold', 'release_gate_rows', 'state_head_target_rows', 'drawdown_control_rows']].copy()
source_cov = source_cov.rename(columns={
    'release_gate_rows': 'source_release_gate_rows',
    'state_head_target_rows': 'source_state_head_target_rows',
    'drawdown_control_rows': 'source_drawdown_control_rows',
})
repair_delta = coverage.merge(source_cov, on='fold', how='left')
for col in ['source_release_gate_rows', 'source_state_head_target_rows', 'source_drawdown_control_rows']:
    repair_delta[col] = pd.to_numeric(repair_delta[col], errors='coerce').fillna(0).astype(int)
repair_delta['delta_release_like_rows'] = repair_delta['release_or_aux_rows'] - (
    repair_delta['source_release_gate_rows'] + repair_delta['source_state_head_target_rows']
)
repair_delta['delta_drawdown_control_rows'] = repair_delta['drawdown_control_rows'] - repair_delta['source_drawdown_control_rows']
repair_delta.to_csv(CARD / 'repair_delta_vs_source.csv', index=False)
print(repair_delta[['contract_variant','fold','release_or_aux_rows','delta_release_like_rows','drawdown_control_rows','delta_drawdown_control_rows','coverage_gate']].to_string(index=False))


    contract_variant               fold  release_or_aux_rows  delta_release_like_rows  drawdown_control_rows  delta_drawdown_control_rows  coverage_gate
        strict_joint      wf_2020_train                 1100                        0                    976                          195           True
        strict_joint      wf_2022_train                    9                        0                     25                           25          False
        strict_joint wf_2023_2024_train                  268                        0                     35                           35           True
aux_state_head_joint      wf_2020_train                 1100                        0                    976                          195           True
aux_state_head_joint      wf_2022_train                  347                      338                     25                           25           True
aux_state_head_joint wf_2023_2024_train                  723                      

In [9]:

def variant_gate(variant: str) -> dict[str, Any]:
    cov = coverage[coverage['contract_variant'].eq(variant)]
    audit = audit_summary[audit_summary['contract_variant'].eq(variant)].iloc[0]
    bc = bc_readiness[bc_readiness['profile'].eq(variant)]
    affected_cov = cov[cov['fold'].isin(AFFECTED_FOLDS)]
    coverage_gate = bool(not affected_cov.empty and affected_cov['coverage_gate'].astype(bool).all())
    scope_gate = bool(audit['scope_training_allowed'])
    lift_gate = bool(audit['lift_training_allowed'])
    bc_gate = bool(False if bc.empty else bc.iloc[0]['bc_readiness_gate'])
    return {
        'contract_variant': variant,
        'coverage_gate': coverage_gate,
        'scope_gate': scope_gate,
        'lift_gate': lift_gate,
        'bc_readiness_gate': bc_gate,
        'target_contract_gate': bool(coverage_gate and scope_gate and lift_gate),
        'affected_fold_coverage': affected_cov[['fold','drawdown_control_rows','release_or_aux_rows','coverage_gate']].to_dict('records'),
    }

strict_gate = variant_gate('strict_joint')
aux_gate = variant_gate('aux_state_head_joint')
selected_variant = 'aux_state_head_joint' if aux_gate['target_contract_gate'] and aux_gate['bc_readiness_gate'] else 'strict_joint'
selected_gate = aux_gate if selected_variant == 'aux_state_head_joint' else strict_gate
bc_summary = summarize_state_objective_bc_readiness(bc_readiness)

if selected_variant == 'aux_state_head_joint' and selected_gate['target_contract_gate'] and selected_gate['bc_readiness_gate']:
    decision = 'release16_joint_drawdown_aux_state_head_target_contract_ready_for_bc_screen_no_promotion'
    next_mechanism = 'run_release16_joint_drawdown_aux_state_head_bc_holdout_slice_screen_official_replay'
    primary_blocker = 'target_revision_ready'
    bc_screen_allowed_next = True
elif strict_gate['target_contract_gate'] and strict_gate['bc_readiness_gate']:
    decision = 'release16_strict_joint_drawdown_release_target_contract_ready_for_bc_screen_no_promotion'
    next_mechanism = 'run_release16_strict_joint_drawdown_release_bc_holdout_slice_screen_official_replay'
    primary_blocker = 'target_revision_ready'
    bc_screen_allowed_next = True
else:
    decision = 'release16_joint_drawdown_release_target_contract_still_sparse_no_training_no_promotion'
    next_mechanism = 'repair_aux_release_state_head_or_action_head_before_bc_replay'
    primary_blocker = 'sparse_slice_target_repair_required'
    bc_screen_allowed_next = False

summary = {
    'experiment': EXPERIMENT,
    'created_at': datetime.now(timezone.utc).isoformat(),
    'decision': decision,
    'promotion_allowed': False,
    'actor_retention_allowed': False,
    'training_allowed_from_this_card': False,
    'retained_training_allowed': False,
    'ppo_allowed_next': False,
    'bc_screen_allowed_next': bool(bc_screen_allowed_next),
    'bc_warm_start_screen_allowed_next': bool(bc_screen_allowed_next),
    'target_contract_gate': bool(selected_gate['target_contract_gate']),
    'bc_readiness_gate': bool(selected_gate['bc_readiness_gate']),
    'next_training_allowed_after_target_revision': bool(bc_screen_allowed_next),
    'primary_blocker': primary_blocker,
    'next_mechanism': next_mechanism,
    'source_card': SPARSE_SOURCE_CARD,
    'source_target_contract': SOURCE_CARD,
    'source_sparse_decision': sparse_summary.get('decision'),
    'canonical_baseline': 'v19reg_v22_budget_usage_bull_guard_v1',
    'direct_comparison_baseline': 'release16_nonconvex_checkpoint_bc_official_replay_v1',
    'reference_baseline': 'soxlbacktest_qlib_buysell_reference',
    'role': 'diagnose',
    'changed_layer': ['target', 'model_selection', 'bc_warm_start'],
    'target_max_dd_pct': TARGET_MAX_DD_PCT,
    'min_sparse_rows': MIN_SPARSE_ROWS,
    'affected_folds': AFFECTED_FOLDS,
    'selected_contract_variant': selected_variant,
    'strict_joint_gate': strict_gate,
    'aux_state_head_joint_gate': aux_gate,
    'bc_summary': bc_summary,
    'guard_fold_note': 'wf_2022_train strict state-head rows remain zero by source guard semantics; aux_state_head_joint is a diagnostic target-revision candidate, not promotion evidence.',
    'repair_requirements': repair_requirements.to_dict('records') if not repair_requirements.empty else [],
    'sidecars': [
        'source_target_coverage.csv',
        'drawdown_control_intervals.csv',
        'drawdown_control_training_targets.csv',
        'strict_joint_target_contract.csv',
        'aux_state_head_release_targets.csv',
        'joint_auxiliary_targets.csv',
        'joint_target_contract.csv',
        'joint_target_coverage.csv',
        'target_scope_audit.csv',
        'target_lift_audit.csv',
        'target_audit_summary.csv',
        'bc_label_distribution.csv',
        'bc_readiness_scores.csv',
        'repair_delta_vs_source.csv',
        'selection.json',
        'summary.json',
        'conclusion.md',
    ],
}
selection = {
    'experiment': EXPERIMENT,
    'created_at': summary['created_at'],
    'decision': decision,
    'selected_contract_variant': selected_variant,
    'promotion_allowed': False,
    'target_contract_gate': bool(summary['target_contract_gate']),
    'bc_readiness_gate': bool(summary['bc_readiness_gate']),
    'bc_screen_allowed_next': bool(summary['bc_screen_allowed_next']),
    'next_training_allowed_after_target_revision': bool(summary['next_training_allowed_after_target_revision']),
    'primary_blocker': primary_blocker,
    'next_mechanism': next_mechanism,
    'source_card': SPARSE_SOURCE_CARD,
    'source_target_contract': SOURCE_CARD,
    'sidecars': summary['sidecars'],
}
write_json(CARD / 'summary.json', summary)
write_json(CARD / 'selection.json', selection)

conclusion_lines = [
    f'# {EXPERIMENT}',
    '',
    f'Decision: `{decision}`.',
    '',
    f'Selected contract variant: `{selected_variant}`.',
    '',
    f'Target contract gate: `{summary["target_contract_gate"]}`. BC readiness gate: `{summary["bc_readiness_gate"]}`.',
    '',
    'Strict joint result: drawdown-control rows are added, but sparse release/state-head coverage remains constrained by guard-fold semantics.',
    '',
    'Aux state-head joint result: guard-fold feature-hit rows are converted into a diagnostic release/rebound target candidate so the next BC holdout screen can test whether slice action fit improves without promoting an actor.',
    '',
    'Risk note: this card does not retain an actor and does not authorize PPO. `wf_2020` path risk still requires downstream checkpoint/reward validation after slice target repair.',
    '',
    f'Next action: `{next_mechanism}`.',
    '',
]
(CARD / 'conclusion.md').write_text('\n'.join(conclusion_lines), encoding='utf-8')
print(json.dumps(json_safe(summary), indent=2, sort_keys=True))


{
  "actor_retention_allowed": false,
  "affected_folds": [
    "wf_2022_train",
    "wf_2023_2024_train"
  ],
  "aux_state_head_joint_gate": {
    "affected_fold_coverage": [
      {
        "coverage_gate": true,
        "drawdown_control_rows": 25,
        "fold": "wf_2022_train",
        "release_or_aux_rows": 347
      },
      {
        "coverage_gate": true,
        "drawdown_control_rows": 35,
        "fold": "wf_2023_2024_train",
        "release_or_aux_rows": 723
      }
    ],
    "bc_readiness_gate": true,
    "contract_variant": "aux_state_head_joint",
    "coverage_gate": true,
    "lift_gate": true,
    "scope_gate": true,
    "target_contract_gate": true
  },
  "bc_readiness_gate": true,
  "bc_screen_allowed_next": true,
  "bc_summary": {
    "decision": "state_objective_bc_ready_for_training",
    "next_mechanism": "run_fold_local_bc_warm_start_then_walkforward_ppo",
    "selected_bc_readiness_gate": true,
    "selected_bc_readiness_loss": 0.0,
    "selected_dominant_a

## Conclusion

The executable conclusion is written to `summary.json`, `selection.json`, `conclusion.md`, and the sidecar CSVs in this card directory after notebook execution. This card is intentionally a target-contract and BC-screen handoff, not a promoted model or retained actor.
